## Data Augmentation
Notebook dedicato alla Data Augmentation



In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split

"""Applica transofrm_heavy alle classi con un numero minore di campioni
    rispetto alla media, transform_basic a tutte le altre."""
class ConditionalDatasetWrapper(Dataset):
    def __init__(self, subset, transform_basic, transform_heavy=None, minority_classes=None):
        self.subset = subset
        self.transform_basic = transform_basic
        self.transform_heavy = transform_heavy
        self.minority_classes = set(minority_classes.tolist() if minority_classes is not None else [])

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform_heavy and y in self.minority_classes:
            x = self.transform_heavy(x)

        elif self.transform_basic:
            x = self.transform_basic(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# 1. Definizione delle Pipeline di Trasformazione delle Immagini
La pipeline (ossia la sequenza di operazioni effettuate su un'immagine) è stata fondata sulle esigenze della rete ResNet (Immagini in ingresso 224x224 pixel su 3 canali (RGB)). Le trasformazioni attuate sono 3:

*   train_heavy (Augmentation Pesante): Dedicata alle classi con meno campioni (minoritarie). Applica modifiche casuali invasive (ritaglio e ridimensionamento, specchiatura, rotazione fino a 15° e alterazione di luminosità/contrasto) per creare artificialmente nuove varianti di immagini rare e insegnare al modello a riconoscerle in contesti diversi.

*   train_basic (Augmentation Leggera): Dedicata alle classi con più campioni (maggioritarie). Ridimensiona l'immagine e applica solo una specchiatura orizzontale casuale.

*   val (Validazione): Non applica alcuna alterazione artificiale. Si limita a ridimensionare e ritagliare l'immagine al centro per adattare i dati ale esigenze della rete (resize + crop + normalize).

Tali trasformazioni sono applicate in modo casuale al fine di fornire una varietà di dati a disposizione il più grande possibile.

# 2. Divisione Stratificata del Dataset (Train/Validation Split)
Caricamento delle immagini dalla cartella indicata usando ImageFolder.
Lo split effettuato è il classico 80-20 (80% Training e 20% Validation).
La divisione è stratificata (stratify=targets), il che significa che sia nel gruppo di train che in quello di validation viene mantenuta rigorosamente la stessa proporzione iniziale di classi.

# 3. Strategia per Risolvere lo Sbilanciamento delle Classi
È il cuore logico della funzione get_dataloaders:
*  Calcola la distribuzione delle immagini per ogni classe (np.bincount) e ne ricava la media.
*  Identifica come classi minoritarie tutte quelle che hanno un numero di campioni inferiore alla media complessiva.
*  Applica il ConditionalDatasetWrapper per smistare le trasformazioni pesanti (train_heavy) solo a queste classi svantaggiate.
*  Calcola i pesi campionari (sample_weights) inversamente proporzionali alla frequenza della classe.
*  Configura un WeightedRandomSampler, uno strumento che dice a PyTorch di estrarre con maggiore probabilità (e con reinserimento: replacement=True) le immagini delle classi rare durante l'addestramento, livellando virtualmente la sbilanciatura del dataset alla radice.



In [ ]:

def get_dataloaders(data_dir, batch_size=32, val_size=0.2, random_seed=42):
  """Carica i dati, applica l'augmentation e restituisce i dataloader."""
  #trasformazioni da applicare
  data_transforms = {
      'train_heavy': transforms.Compose([
          transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # ritaglia una parte casuale dell'immagine (tra l'80% e il 100% dell'immagine)
          transforms.RandomHorizontalFlip(), #specchia l'immagine orizzontalmente con una probabilità del 50%
          transforms.RandomRotation(15), #ruota fino a +15° o -15° l'immagine
          transforms.ColorJitter(brightness=0.2, contrast=0.2), #varia colore e lumonistà per simulare ambienti diversi
          transforms.ToTensor(),
          transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) #adattamento al training originale
      ]),
      'train_basic': transforms.Compose([
          transforms.Resize(256),
          transforms.CenterCrop(224), #porta tutte le immagini alla stessa dimensione standard
          transforms.RandomHorizontalFlip(), #specchia l'immagine orizzontalmente con una probabilità del 50%
          transforms.ToTensor(),
          transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
      ]),
      'val': transforms.Compose([ #non si applica alcuna augmentation: si devono simulare dati mai visti e mai modificati
          transforms.Resize(256),
          transforms.CenterCrop(224), #porta tutte le immagini alla stessa dimensione standard
          transforms.ToTensor(),
          transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
      ]),
  }
  #caricamento del dataset di base
  full_dataset = datasets.ImageFolder(data_dir)
  targets = torch.tensor(full_dataset.targets)
  class_names=full_dataset.classes
  n_classes=len(class_names)
  # Split 80% Train, 20% Val
  train_idx, val_idx = train_test_split(
      np.arange(len(targets)),
      test_size=val_size,
      random_state=random_seed,
      stratify=targets
  )
  #Identificazioni classi con meno campioni
  train_targets= targets[train_idx]
  """'np.bicount' restituisce quante volte appare ciascun target all'interno dei
      dati di training raccolti."""
  class_counts=np.bincount(train_targets, minlength=n_classes)
  avg_samples = class_counts.mean()
  minority_classes_idx = np.where(class_counts < avg_samples)[0]

  print(f"Campioni medi per classe: {avg_samples:.0f}")
  print("Classi minoritarie identificate (sottoposte ad Augmentation):")
  for idx in minority_classes_idx:
      print(f"- {full_dataset.classes[idx]} ({class_counts[idx]} img in train)")
  train_subset= torch.utils.data.Subset(full_dataset, train_idx)
  val_subset=torch.utils.data.Subset(full_dataset, val_idx)


  train_dataset = ConditionalDatasetWrapper(
      train_subset,
      transform_basic=data_transforms['train_basic'],
      transform_heavy=data_transforms['train_heavy'],
      minority_classes=minority_classes_idx
  )

  # Alla validazione passiamo solo la trasformazione base (val)
  val_dataset = ConditionalDatasetWrapper(
      val_subset,
      transform_basic=data_transforms['val']
  )
  class_weights=1.0/class_counts.astype(float)
  sample_weights=class_weights[train_targets] #ad ogni immagine si assegna il peso della sua classe

  """Permette di pescare i dati dal training set seguendo i pesi definiti sopra,
  una stessa immagine può essere ripescata più volte - questo è necessario per le classi con pochi sample"""
  sampler=WeightedRandomSampler(
      weights=torch.tensor(sample_weights, dtype=torch.float),
      num_samples=len(train_dataset),
      replacement=True
  )

  dataloaders = {
      'train': DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True),
      'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
  }
  print(f"\nDataloader pronti.")
  return dataloaders, class_names




3. Caricamento del dataset e split in train e test: lo splt applicato è il classico 80-20, in cui l'80% dei dati viene destinato per il training, mentre il restante 20% viene utilizzato per la validation della rete. Si utilizza uno split stratificato per mantenere le proporzioni tra le classi



```
print("Caricamento dataset e creazione split...")
full_dataset = datasets.ImageFolder(DATA_DIR)
targets = torch.tensor(full_dataset.targets)
class_names=full_dataset.classes
n_classes=len(class_names)

# Split 80% Train, 20% Val
train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.2,
    random_state=42,
    stratify=targets
)

```



4. Si identificano le classi con un numero di campioni minore della media



```
train_targets= targets[train_idx]
"""'np.bicount' restituisce quante volte appare ciascun target all'interno dei
    dati di training raccolti."""
class_counts=np.bincount(train_targets, minlength=n_classes)
media_campioni = class_counts.mean()
minority_classes_idx = np.where(class_counts < media_campioni)[0]

print(f"Campioni medi per classe: {media_campioni:.0f}")
print("Classi minoritarie identificate (sottoposte ad Augmentation):")
for idx in minority_classes_idx:
    print(f"- {full_dataset.classes[idx]} ({class_counts[idx]} img in train)")
```



5. Si procede con la creazione dei singoli dataset: per i dati di training si applicano entrambe le tipologie di trasformazioni (heavy e basic). Per i dati di validation, invece, non si applica alcuna trasformazione: i dati vengono semplicemente adattati ai layer convoluzionali. I dati di training devono essere in grado di simulare il comportamento rreale della rete.



```
train_subset= torch.utils.data.Subset(full_dataset, train_idx)
val_subset=torch.utils.data.Subset(full_dataset, val_idx)


train_dataset = ConditionalDatasetWrapper(
    train_subset,
    transform_basic=data_transforms['train_basic'],
    transform_heavy=data_transforms['train_heavy'],
    minority_classes=minority_classes_idx
)

# Alla validazione passiamo solo la trasformazione base (val)
val_dataset = ConditionalDatasetWrapper(
    val_subset,
    transform_basic=data_transforms['val']
)
```



Si introduce un sampler che associa a ciascuna classe un peso inversamente proporzionale alla frequenza con cui la classe compare nel dataset: più una clase è frequente minore sarà il peso. Questo peso influirà sulla probabilità di pescare una determinata classe dal dataset.



```
class_weights=1.0/class_counts.astype(float)
sample_weights=class_weights[train_targets] #ad ogni immagine si assegna il peso della sua classe

"""Permette di pescare i dati dal training set seguendo i pesi definiti sopra,
 una stessa immagine può essere ripescata più volte - questo è necessario per le classi con pochi sample"""
sampler=WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_dataset),
    replacement=True
)

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True),
}

print(f"\nDataloader pronti.")

```



# Data Augmentation Uniforme per tutte le classi

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

class StandardDatasetWrapper(Dataset):
    """Wrapper semplice per applicare la stessa trasformazione a tutto il subset."""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


def get_uniform_dataloaders(data_dir, batch_size=32, val_size=0.2, random_seed=42):
    """
    Carica i dati applicando la STESSA data augmentation a tutte le classi.

    VERSIONE PREDISPOSTA PER ABLATION STUDY:
    Eliminate completamente le alterazioni di luminosità e contrasto per rimuovere
    il rumore artificiale che creava falsi positivi sulla classe metallo.
    Preservate solo le variazioni geometriche pulite e la saturazione/tinta per i trasparenti.
    """

    # Pipeline alleggerita e ripulita dal rumore di contrasto/luminosità
    data_transforms = {
        'train': transforms.Compose([
            transforms.Resize(256),

            # Invarianza di scala e ritaglio geometrico pulito
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),

            # Invarianza speculare completa (coerente con la caduta nei cassonetti)
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(15),

            # --- CROMATICA PULITA (SENZA LUMINOSITÀ E CONTRASTO) ---
            # Brightness e Contrast azzerati (valore = 0 significa nessuna variazione).
            # Mantenute saturazione e tinta per evitare che vetro e plastica si ancorino
            # cromaticamente allo sfondo fisso delle foto.
            transforms.ColorJitter(
                brightness=0.0,
                contrast=0.0,
                saturation=0.3,
                hue=0.05
            ),

            transforms.ToTensor(),

            # Normalizzazione standard per modelli ImageNet
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
        'val': transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }

    # Caricamento del dataset di base
    full_dataset = datasets.ImageFolder(data_dir)
    targets = torch.tensor(full_dataset.targets)
    class_names = full_dataset.classes

    # Split Train e Val stratificato
    train_idx, val_idx = train_test_split(
        np.arange(len(targets)),
        test_size=val_size,
        random_state=random_seed,
        stratify=targets
    )

    train_subset = torch.utils.data.Subset(full_dataset, train_idx)
    val_subset = torch.utils.data.Subset(full_dataset, val_idx)

    # Applichiamo i wrapper con le trasformazioni uniformi
    train_dataset = StandardDatasetWrapper(train_subset, transform=data_transforms['train'])
    val_dataset = StandardDatasetWrapper(val_subset, transform=data_transforms['val'])

    # Configurazione dei Dataloader ottimizzati per GPU
    dataloaders = {
        'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
        'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    }

    print(f"\nDataloader in assetto 'Clean' pronti per il test di controllo.")
    print(f"Rimossi: Variazioni di luminosità, contrasto, sfocatura e distorsione prospettica.")
    return dataloaders, class_names